# 8. Models del llenguatge (amb n-grames)

Tractament de Dades Textuals i Codificades — Eixample Clínic, 2026

Pol Pastells, ppastells@eixampleclinic.es

---

Model generació noms amb N-grames (inspirat per [Andrey Karpathy](https://youtu.be/PaCmpygFfXo?si=jRAnbQPElsccgK15).
Per informació més teòrica vegeu el capítol 3 del llibre de [Jurafsky i Martin](https://web.stanford.edu/~jurafsky/slp3/))

Partirem de dades de noms de Catalunya, extretes de https://www.idescat.cat/noms/. Les dades ja han estat [prèviament netejades](https://github.com/Pastells/python_avan/blob/main/3.1.solucions-exemple_pandas.ipynb) (estan en minúscules i sense accents).

## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 100
%matplotlib inline
%config InlineBackend.figure_format='retina'

Tenim un fitxer amb noms a Catalunya, segons la seva freqüència

In [ ]:
df = pd.read_csv("noms_cat.csv", keep_default_na=False)
df.head()

Volem fer un model que generi noms, aprenent dels patrons d'aquesta llista.
Treballarem a nivell de caràcters, farem servir `#` per marcar inici i fi dels noms.

In [ ]:
llista_noms = df["Nom"]
noms = [f"#{nom}#" for nom in llista_noms]
noms[:5]

Creem una llista amb els caràcters possibles i diccionaris per passar de caràcter a enter i al revés. És a dir, fem un _encoding_  dels caràcters amb nombres enters

In [ ]:
caracters = ["#"] + sorted(list(set("".join(llista_noms))))
ncars = len(caracters)
c2i = {c: i for i, c in enumerate(caracters)}
i2c = {i: c for i, c in enumerate(caracters)}
c2i

## Distribució de caràcters (llei de Zipf)

In [ ]:
N1 = np.zeros(ncars, dtype=np.int32)
for nom in noms:
    for c in nom:
        N1[c2i[c]] += 1

P1 = N1 / N1.sum()
sorted_indices = np.argsort(P1)[::-1]

P1 = P1[sorted_indices]
caracters_sorted = [caracters[i] for i in sorted_indices]

plt.scatter(caracters_sorted, np.log(P1))
plt.show()

## Model amb recompte de bigrames

Podem fer servir zip per crear una llista de bigrames

In [ ]:
noms[0], list(zip(noms[0], noms[0][1:]))

Agafant la llista de noms, fem una matriu amb totes les possiblitats de dos caràcters, per tal de fer un recompte. Al principi crearem una matriu plena de zeros i l'anirem omplint.

In [ ]:
N = np.zeros((ncars, ncars), dtype=int)

Emplenem la matriu amb el nombre de vegades que apareix cada combinació i ho visualitzem

In [ ]:
for nom in noms:
    for c1, c2 in zip(nom, nom[1:]):
        ix1 = c2i[c1]
        ix2 = c2i[c2]
        N[ix1, ix2] += 1

In [ ]:
plt.figure(figsize=(25, 25))
plt.imshow(N, cmap="Blues")
for i in range(len(i2c)):
    for j in range(len(i2c)):
        cstr = i2c[i] + i2c[j]
        plt.text(j, i, cstr, ha="center", va="bottom")
        plt.text(j, i, N[i, j].item(), ha="center", va="top")

Anem a generar nous noms a partir d'aquestes freqüències.

Per cada nom que generem, comencem amb '#' i mirem quina probabilitat hi ha per la següent caràcter, fins que ens surti un altre '#' final.

N[0] (fila 0 de la matriu N, com podem veure a la figura superior) ens diu el recompte dels caràcters que segueixen '#'. Per exemple veiem que no va mai seguit d'un altre '#', espai o apòstrof.

In [ ]:
N[0]

Si ens interessa, podem fer un gràfic de quins caràcters solen seguir quins altres, o predecedir-los (canvieu `N[i]` per `N[:, i]`)

In [ ]:
c = "a"
i = c2i[c]
plt.bar(np.arange(ncars), N[i])
plt.title(f"'{i2c[i]}' va seguit de...")
for i, value in enumerate(N[i]):
    plt.text(i, value, i2c[i], ha="center", va="bottom")

El que volem és **normalitzar** aquest recompte per obtenir freqüències, que interpretarem com a probabilitats

In [ ]:
N[0] / N[0].sum()

Com que haurem de fer aquesta operació per totes les possibles combinacions creem una nova matriu P amb les files normalitzades

In [ ]:
N.sum(axis=1, keepdims=True).shape  # vector columna

In [ ]:
P = N / N.sum(axis=1, keepdims=True)  # normalitzem

Comprovem que la fila és la mateixa

In [ ]:
print(P[0] == N[0] / N[0].sum())
P[0]

In [ ]:
# P = np.ones((ncars, ncars)) / ncars  # comparació amb model aleatori (per executar més endavant)

Molt bé, ja tenim la matriu de probabilitats de bigrames. En teoria amb això ja podem fer un primer model que generi noms seguint aquestes freqüències

In [ ]:
# genera 10 noms aleatoris
for i in range(10):
    ix = 0  # índex del primer caràcter '#'
    nom = ""
    while True:
        ix = np.argmax(np.random.multinomial(1, P[ix]))
        if ix == 0:
            break
        nom += i2c[ix]
    print(nom)

Són molt dolents!

Es veu que fer servir bigrames no ens dona resultats massa bons.

Ara bé, si ho comparem amb una distribució totalment aleatòria, donant el mateix pes a totes les combinacions, no és tant horrible. Així que estem fent algo bé.
Comprova-ho descomentant la cel·la de dalt amb `np.ones` i refent la generació.

## Ho podem fer millor?

Per ara hem considerat tots els noms de la mateixa manera. Podem incorporar la freqüència que apareix a les dades originals per fer que els bigrames més comuns apareguin més.

In [ ]:
freqs = df.freq.to_numpy()
freqs

In [ ]:
N2 = np.zeros((ncars, ncars), dtype=int)
for i, nom in enumerate(noms):
    for c1, c2 in zip(nom, nom[1:]):
        ix1 = c2i[c1]
        ix2 = c2i[c2]
        N2[ix1, ix2] += freqs[i]

In [ ]:
P2 = N2 / N2.sum(axis=1, keepdims=True)

In [ ]:
for i in range(10):
    ix = 0
    nom = ""
    while True:
        ix = np.argmax(np.random.multinomial(1, P[ix]))
        p = P[ix]
        ix = np.random.choice(len(p), p=p)
        if ix == 0:
            break
        nom += i2c[ix]
    print(nom)

Potser són una mica millors(?), però molts segueixen siguent dolents. Considerar només dos caràcters és molt limitant

## Trigrames

Fins ara hem fet els bigrames amb un simple zip de python.

Ara passem a fer servir la funció `ngrams` de `nltk` per no complicar-nos la vida

In [ ]:
from nltk import ngrams

In [ ]:
N3 = np.zeros((ncars, ncars, ncars), dtype=np.int32)
for i, nom in enumerate(noms):
    for c1, c2, c3 in ngrams(nom, 3):
        N3[c2i[c1], c2i[c2], c2i[c3]] += freqs[i]

Ara cal fixar-nos en els 2 caràcters anteriors per generar el següent. També farem ús del model de bigrames per generar el primer caràcter de tots.

Hem de normalitzar el recompte de caràcters que segueixen qualsevols dos caràcters, per exemple "ma"

In [ ]:
c2i["m"], c2i["a"]

In [ ]:
N3[15, 3]

In [ ]:
plt.bar(np.arange(ncars), N3[15, 3])
for i, value in enumerate(N3[15, 3]):
    plt.text(i, value, i2c[i], ha="center", va="bottom")

In [ ]:
N3[15, 3] / N3[15, 3].sum()

Com haviem fet pels bigrames, fem totes les operacions de cop

In [ ]:
N3.sum(axis=2, keepdims=True).shape

In [ ]:
P3 = N3 / N3.sum(axis=2, keepdims=True)
P3[0, 0]

Veiem que ens surt un error. És degut a una divisió entre 0, que dona `nan` com a resultat.

Podríem no fer-ne cas, ja que precisament no farem servir les combinacions que mai apareixen. El que farem, però, és fer servir `np.divide`, que ens permetrà obtenir zeros en comptes de `nan`.

In [ ]:
N3_sum = N3.sum(axis=2, keepdims=True)
P3 = np.divide(N3, N3_sum, out=np.zeros(N3.shape, dtype=float), where=N3_sum != 0)
P3[0, 0]

I comprovem que el resultat és el mateix que a dalt

In [ ]:
P3[15, 3]

Fins ara hem generat resultats aleatoris no reproduïbles, si volem fer un experiment o obtenir dades per un article, és bona idea **fixar una llavor (seed), que ens permeti reproduir els resultats**.

Segons d'on provingui l'aleatorietat caldrà veure com es fixa la llavor. Per numpy es fa amb `np.random.seed`

In [ ]:
np.random.seed(42)

In [ ]:
for i in range(10):
    # Primer caràcter a partir de bigrames
    ix1 = 0
    ix2 = np.argmax(np.random.multinomial(1, P2[ix1]))
    nom = i2c[ix2]

    # Resta amb trigrames
    while True:
        ix3 = np.argmax(np.random.multinomial(1, P3[ix1, ix2]))
        if ix3 == 0:
            break
        nom += i2c[ix3]
        ix1 = ix2
        ix2 = ix3

    print(nom)

## Exercici A - Continuació amb noms de persona

### Exercici A1: adapta el codi de generació de noms amb trigrames perquè generi noms d'home o de dona

Crea una funció `genera_noms` que rebi un sol argument `sexe` i retorni un únic nom

Deixarem les matrius de probabilitat com a variables globals.

Et caldran dues matrius per homes (PH2, PH3) i dues per dones (PD2, PD3).


### Exercici A2: amplia la funció `genera_noms` perquè faci servir 4-grames

###   *Exercici A3: amplia la funció `genera_noms` perquè faci servir n-grames

Com que generar les matrius de probabilitat pot ser un procés lent, busca com es fa servir el paquet `tqdm` per mostrar una barra amb el progrés.

Fes el codi en general, i prova'l amb 5-grames.

1. Fes primer una funció `get_Pn` que retorni les matrius de probabilitat corresponents a n-grames, partint de `get_P4`.
2. Genera i guarda les matrius de P2 a P5 corresponents a homes i dones en un vector per cada sexe (PnsH, PnsD).
3. Crea la funció `genera_noms(Pns, n)`, que rebi com a input Pns i el nombre d'ngrames que farà servir per la generació. 
 

### Pistes

1. Per `genera_noms`, pots fer servir `deque` per guardar els indexs ix1, ix2... en un array que automàticament mantingui només `n-1` índexs.

In [ ]:
from collections import deque

ixs = deque([0], maxlen=3)
ixs.append(1)
print(ixs)
ixs.append(2)
print(ixs)
ixs.append(15)
print(ixs)
ixs.append(3)
print(ixs)
ixs.append(14)
print(ixs)

2. Pots indexar un array de numpy amb una tuple, si ho fas directament amb deque (igual que amb una llista o un array, ho tractarà com a índexs diferents)

In [ ]:
PH4[tuple(ixs)]

3. Fixa't en què es repeteix a `genera_noms` de l'exercici 2. Agrupa-ho dins un `while True` amb un únic `return nom`

## Exercici B - Principis actius

Ara proveu a canviar les dades a una llista de principis actius. Ens podem baixar un excel amb medicacions de l'AEMPS: https://listadomedicamentos.aemps.gob.es/Medicamentos.xls

In [ ]:
# pot ser que us calgui instalar aquesta llibreria per poder llegir excel
# ! pip install openpyxl

In [ ]:
medicaments = pd.read_excel("Medicamentos.xls")

In [ ]:
# cada medicament té una columna amb una llista dels principis actius
medicaments["Principios Activos"]

In [ ]:
# podem passar-ho a llista
medicaments["Principios Activos"].str.split(", ").head()

In [ ]:
# i aplanar-ho
from itertools import chain

principis = list(chain.from_iterable(medicaments["Principios Activos"].str.split(", ").to_list()))
len(principis), principis[:20]

In [ ]:
from collections import Counter

principis_count = Counter(principis)
len(principis_count), principis_count.most_common(10)

In [ ]:
llista_noms = list(principis_count.keys())
llista_noms = [nom.lower() for nom in llista_noms]

noms = [f"#{nom}#" for nom in llista_noms]
freqs = np.array(list(principis_count.values()))

caracters = ["#"] + sorted(list(set("".join(llista_noms))))
ncars = len(caracters)
c2i = {c: i for i, c in enumerate(caracters)}
i2c = {i: c for i, c in enumerate(caracters)}

In [ ]:
# Quants principis únics tenim?
print(f"Total entrades: {len(principis)}")
print(f"Principis únics: {len(principis_count)}")
print(f"Caràcters únics: {ncars}")

# Quins caràcters nous apareixen vs els noms catalans?
print(caracters)